In [67]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [68]:
import json
import pandas as pd
import numpy as np
import requests
import time
import qdrant_client
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from sklearn.metrics import ndcg_score
from qdrant_client import QdrantClient
from qdrant_client.http.exceptions import UnexpectedResponse

In [69]:
# Load embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [70]:
QDRANT_URL = "http://192.68.10.50:6333"
client = QdrantClient(url=QDRANT_URL)

In [126]:
queries = {q["_id"]: q["text"] for q in map(json.loads, open("queries.jsonl"))}
corpus = {c["_id"]: c["text"] for c in map(json.loads, open("SciFact.jsonl"))} # corpus.jsonl
qrels = pd.read_csv("test.tsv", sep="\t", names=["query-id", "corpus-id", "score"]) #qrels/test.tsv

In [127]:
# print(qrels["query-id"].unique()[:11])

In [128]:
# Select all queries for evaluation
selected_queries = qrels["query-id"]

In [129]:
# Remove duplicate queries
selected_queries = list(set(selected_queries))

In [130]:
print(selected_queries)

['723', '960', '700', '1362', '691', '535', '1379', '781', '232', '536', '684', '514', '660', '1202', '1389', '832', '1191', '1024', '218', '1270', '903', 'query-id', '692', '911', '521', '1179', '478', '800', '956', '1089', '1', '54', '805', '56', '124', '42', '577', '613', '814', '1049', '36', '1320', '236', '50', '649', '887', '70', '729', '967', '137', '528', '410', '637', '744', '1221', '1232', '539', '388', '142', '1041', '1290', '516', '248', '1382', '208', '261', '1282', '693', '820', '636', '1021', '1273', '830', '275', '921', '437', '411', '756', '882', '354', '1344', '831', '233', '793', '1121', '452', '230', '985', '1088', '1104', '1110', '1298', '1266', '386', '1271', '298', '821', '1200', '1316', '1274', '491', '300', '1272', '1279', '180', '312', '811', '957', '338', '1012', '569', '533', '575', '619', '628', '163', '593', '475', '1204', '133', '975', '100', '532', '784', '217', '525', '993', '57', '185', '324', '1207', '1385', '3', '1099', '1185', '99', '1352', '1363', 

In [73]:
# def get_relevant_answer_rank(retrieved_docs, relevant_docs):
#     """Get rank of first relevant document (0-9)"""
#     for i, doc_id in enumerate(retrieved_docs):
#         if doc_id in relevant_docs:
#             return i
#     return 9 

In [74]:
# def retrieve_with_rank(query_id, query_text):
#     """Retrieve docs and get rank for one iteration"""
#     try:
#         # Get query vector
#         query_vector = model.encode(query_text).tolist()
        
#         # Get top 10 results
#         results = client.search(
#             collection_name="scifact", 
#             query_vector=query_vector,
#             limit=10
#         )
        
#         # Extract doc IDs
#         retrieved_docs = [r.payload["id"] for r in results]
        
#         # Get relevant docs for this query
#         relevant_docs = set(qrels[qrels["query-id"] == query_id]["corpus-id"])
        
#         # Get rank
#         rank = get_relevant_answer_rank(retrieved_docs, relevant_docs)
        
#         return rank
#     except Exception as e:
#         print(f"Error for query {query_id}: {e}")
#         return None


In [75]:
# # Main execution
# iterations = 3
# results = []

In [76]:
# # Get all query IDs
# all_queries = qrels["query-id"]

# # Process each query
# for query_id in all_queries:
#     query_text = queries.get(str(query_id), "")
    
#     row = {
#         "Query ID": query_id,
#         "relevant_answer_rank_v1": None,
#         "relevant_answer_rank_v2": None,
#         "relevant_answer_rank_v3": None
#     }
    
#     # Run 3 iterations
#     for i in range(iterations):
#         rank = retrieve_with_rank(query_id, query_text)
#         row[f"relevant_answer_rank_v{i+1}"] = rank
    
#     # Check determinism
#     ranks = [row[f"relevant_answer_rank_v{i+1}"] for i in range(iterations)]
#     row["deterministic"] = "Yes" if len(set(ranks)) == 1 else "No"
    
#     results.append(row)

# # Create DataFrame
# df = pd.DataFrame(results)


In [77]:
# # Remove row if it contains "query-id" as value
# print("\nDuplicate counts:")
# print(df['Query ID'].value_counts())
# df = df[df['Query ID'] != 'query-id']

In [50]:
df

,Query ID,relevant_answer_rank_v1,relevant_answer_rank_v2,relevant_answer_rank_v3,deterministic
2,1,4.0,4.0,4.0,Yes
3,3,2.0,2.0,2.0,Yes
4,5,1.0,1.0,1.0,Yes
5,13,9.0,9.0,9.0,Yes
6,36,2.0,2.0,2.0,Yes
...,...,...,...,...,...
637,1379,0.0,0.0,0.0,Yes
638,1382,0.0,0.0,0.0,Yes
639,1385,1.0,1.0,1.0,Yes
640,1389,0.0,0.0,0.0,Yes


In [135]:
def retrieve_top_k(query_text, k=10):
    """Retrieve top-k results from Qdrant for a given query."""
    query_vector = model.encode(query_text).tolist()
    results = client.search(collection_name="scifact", query_vector=query_vector, limit=k)
    return [r.payload["id"] for r in results]  # Retrieve document IDs

In [136]:
# def compute_metrics(retrieved_docs, relevant_docs):
#     """Compute DCG, NDCG, Recall@10, and MAP@10."""
#     relevance_scores = [1 if doc in relevant_docs else 0 for doc in retrieved_docs]
    
#     dcg = sum(rel / np.log2(i+2) for i, rel in enumerate(relevance_scores))
#     ideal_dcg = sum(1 / np.log2(i+2) for i in range(len(relevant_docs)))
#     ndcg = dcg / ideal_dcg if ideal_dcg > 0 else 0
    
#     recall_at_10 = sum(relevance_scores) / len(relevant_docs)
    
#     precisions = [sum(relevance_scores[:i+1]) / (i+1) for i in range(len(relevance_scores)) if relevance_scores[i] > 0]
#     map_at_10 = np.mean(precisions) if precisions else 0
    
#     return dcg, ndcg, recall_at_10, map_at_10


def compute_metrics(retrieved_docs, relevant_docs):
    """Compute DCG, NDCG, Recall@10, and MAP@10 for a single query."""
    relevance_scores = [1 if doc in relevant_docs else 0 for doc in retrieved_docs]
    
    dcg = sum(rel / np.log2(i+2) for i, rel in enumerate(relevance_scores))
    ideal_dcg = sum(1 / np.log2(i+2) for i in range(len(relevant_docs)))
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else 0
    
    recall_at_10 = sum(relevance_scores) / len(relevant_docs) if relevant_docs else 0
    
    precisions = [sum(relevance_scores[:i+1]) / (i+1) for i in range(len(relevance_scores)) if relevance_scores[i] > 0]
    map_at_10 = np.mean(precisions) if precisions else 0
    
    return dcg, ndcg, recall_at_10, map_at_10


In [11]:
# Run multiple evaluations to check determinism
iterations = 3  # Can be increased to 5 later
results = []

In [137]:
for query_id in selected_queries:
    query_text = queries.get(query_id, "")
    relevant_docs = set(qrels[qrels["query-id"] == query_id]["corpus-id"])

In [ ]:
# 1. Test server availability first
import requests
import time
from qdrant_client import QdrantClient
from qdrant_client.http.exceptions import UnexpectedResponse

def test_qdrant_connection():
    try:
        response = requests.get("http://localhost:6333/collections")
        print(f"Server connection test: {response.status_code}")
        return True
    except:
        return False

# 2. Initialize client with better settings
if test_qdrant_connection():
    client = QdrantClient(
        url="http://localhost:6333",
        timeout=60.0,  # Increased timeout
        prefer_grpc=False
    )
    
    # 3. Try scrolling with error handling
    try:
        result = client.scroll(
            collection_name="scifact",
            limit=1,
            timeout=30
        )
        print(result)
    except Exception as e:
        print(f"Error during scroll: {e}")
        # print("Try using IP 127.0.0.1 instead of localhost")
else:
    print("Could not connect to Qdrant server")

Server connection test: 200
([Record(id=0, payload={'id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. 

In [139]:
print(client.scroll(collection_name="scifact", limit=1))

([Record(id=0, payload={'id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the

In [140]:
# test_query = "What is the effect of vaccines?"
# retrieved_docs = retrieve_top_k(test_query, k=5)
# print(retrieved_docs)


In [141]:
# test_query = "0-dimensional biomaterials lack inductive properties"
# retrieved_docs = retrieve_top_k(test_query, k=5)
# print(retrieved_docs)

In [142]:
# # Add after model initialization and before DataFrame creation
# def get_relevant_content_rank(query_text, retrieved_docs):
#     """Calculate cosine similarity rank for relevant content"""
#     query_vector = model.encode(query_text)
    
#     # Get text content for retrieved docs
#     doc_texts = [corpus.get(doc_id, "") for doc_id in retrieved_docs]
#     doc_vectors = model.encode(doc_texts)
    
#     # Calculate cosine similarities
#     similarities = []
#     for doc_vector in doc_vectors:
#         similarity = np.dot(query_vector, doc_vector) / (np.linalg.norm(query_vector) * np.linalg.norm(doc_vector))
#         similarities.append(similarity)
    
#     # Return average similarity as rank
#     return np.mean(similarities) if similarities else 0

In [143]:
print(f"First few queries: {selected_queries[:5]}")
print(f"Total queries before deduplication: {len(selected_queries)}")


First few queries: ['723', '960', '700', '1362', '691']
Total queries before deduplication: 301


In [144]:
selected_queries = list(set(qrels["query-id"].tolist()))
print(f"Total unique queries after deduplication: {len(selected_queries)}")


Total unique queries after deduplication: 301


In [145]:
print(f"Total unique queries after deduplication: {len(selected_queries)}")

Total unique queries after deduplication: 301


In [146]:
# def safe_retrieve_top_k(query_text, k=10, max_retries=3):
#     """Wrapper for retrieve_top_k with error handling"""
#     for attempt in range(max_retries):
#         try:
#             return retrieve_top_k(query_text, k)
#         except (requests.exceptions.ConnectionError, UnexpectedResponse) as e:
#             print(f"Attempt {attempt + 1}/{max_retries} failed: {e}")
#             if attempt == max_retries - 1:
#                 raise
#             time.sleep(1)  # Wait before retry

# # Modified query loop
# for query_id in selected_queries:
#     query_text = queries.get(query_id, "")
#     relevant_docs = set(qrels[qrels["query-id"] == query_id]["corpus-id"])
    
#     row = {"Query ID": query_id}
#     for i in range(iterations):
#         try:
#             retrieved_docs = safe_retrieve_top_k(query_text, k=10)
#             dcg, ndcg, recall, map_ = compute_metrics(retrieved_docs, relevant_docs)
#             row[f"Iteration {i+1}"] = f"NDCG: {ndcg:.3f}, Recall@10: {recall:.3f}, MAP@10: {map_:.3f}"
            
#             # Calculate relevant content rank
#             # rank = get_relevant_content_rank(query_text, retrieved_docs)
#             # row["relevant_content_rank"] = f"{rank:.3f}"
            
#         except Exception as e:
#             print(f"Error processing query {query_id}, iteration {i+1}: {e}")
#             row[f"Iteration {i+1}"] = "Error: Failed to retrieve results"
#             row["relevant_content_rank"] = "Error"
    
#     results.append(row)



def safe_retrieve_top_k(query_text, k=10, max_retries=3):
    """Wrapper for retrieve_top_k with error handling."""
    for attempt in range(max_retries):
        try:
            return retrieve_top_k(query_text, k)
        except Exception as e:
            print(f"Attempt {attempt + 1}/{max_retries} failed: {e}")
            if attempt == max_retries - 1:
                raise
            time.sleep(1)  # Wait before retrying

# Initialize overall metric accumulators
total_dcg, total_ndcg, total_recall, total_map = 0, 0, 0, 0
valid_queries = 0  # Count queries with valid retrieval


# Process all selected queries
for query_id in selected_queries:
    query_text = queries.get(query_id, "")
    relevant_docs = set(qrels[qrels["query-id"] == query_id]["corpus-id"])
    
    try:
        retrieved_docs = safe_retrieve_top_k(query_text, k=10)
        dcg, ndcg, recall, map_ = compute_metrics(retrieved_docs, relevant_docs)
        
        total_dcg += dcg
        total_ndcg += ndcg
        total_recall += recall
        total_map += map_
        valid_queries += 1
    
    except Exception as e:
        print(f"Error processing query {query_id}: {e}")

# Compute overall averages
if valid_queries > 0:
    avg_dcg = total_dcg / valid_queries
    avg_ndcg = total_ndcg / valid_queries
    avg_recall = total_recall / valid_queries
    avg_map = total_map / valid_queries
else:
    avg_dcg = avg_ndcg = avg_recall = avg_map = 0

# Print final results
print(f"Overall Metrics Across {valid_queries} Queries:")
print(f"  🔹 DCG: {avg_dcg:.3f}")
print(f"  🔹 NDCG: {avg_ndcg:.3f}")
print(f"  🔹 Recall@10: {avg_recall:.3f}")
print(f"  🔹 MAP@10: {avg_map:.3f}")



C:\Users\BeloAbhigyan\AppData\Local\Temp\ipykernel_14628\3409790786.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(collection_name="scifact", query_vector=query_vector, limit=k)


Overall Metrics Across 301 Queries:
  🔹 DCG: 0.688
  🔹 NDCG: 0.643
  🔹 Recall@10: 0.781
  🔹 MAP@10: 0.597


In [83]:
df = pd.DataFrame(results)

In [20]:
df.head

<bound method NDFrame.head of     Query ID                                   Iteration 1  \
0   query-id  NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000   
1          1  NDCG: 0.387, Recall@10: 1.000, MAP@10: 0.200   
2          3  NDCG: 0.500, Recall@10: 1.000, MAP@10: 0.333   
3          5  NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500   
4         13  NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000   
5         36  NDCG: 0.491, Recall@10: 1.000, MAP@10: 0.278   
6         42  NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500   
7         48  NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000   
8         49  NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000   
9         50  NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000   
10        51  NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000   

   relevant_content_rank                                   Iteration 2  \
0                  0.106  NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000   
1                  0.294  NDCG: 0.387, Recall@10: 1.000, MAP@10: 0.200   
2  

In [84]:
print("DataFrame Info:")
print(df.info())

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 340 entries, 0 to 339
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Query ID  340 non-null    object
 1   Metrics   340 non-null    object
dtypes: object(2)
memory usage: 5.4+ KB
None


In [118]:
print("\nDuplicate counts:")
print(df['Query ID'].value_counts())


Duplicate counts:


KeyError: 'Query ID'

In [23]:
df['deterministic'] = ''

In [87]:
df

,Query ID,Metrics
0,query-id,"NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000"
1,1,"NDCG: 0.387, Recall@10: 1.000, MAP@10: 0.200"
2,3,"NDCG: 0.500, Recall@10: 1.000, MAP@10: 0.333"
3,5,"NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500"
4,13,"NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000"
...,...,...
335,1379,"NDCG: 0.956, Recall@10: 1.000, MAP@10: 0.887"
336,1382,"NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000"
337,1385,"NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500"
338,1389,"NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000"


In [119]:
# Remove row if it contains "query-id" as value
print("\nDuplicate counts:")
print(df['Query ID'].value_counts())
df = df[df['Query ID'] != 'query-id']


Duplicate counts:


KeyError: 'Query ID'

In [108]:
df = df.drop_duplicates(subset=['query-id'])

In [112]:
print("\nDuplicate counts:")
print(df['query-id'].value_counts())


Duplicate counts:
query-id
1       1
957     1
936     1
922     1
921     1
       ..
501     1
491     1
478     1
475     1
1395    1
Name: count, Length: 300, dtype: int64


In [109]:
df

,query-id,Metrics
1,1,"NDCG: 0.387, Recall@10: 1.000, MAP@10: 0.200"
2,3,"NDCG: 0.500, Recall@10: 1.000, MAP@10: 0.333"
3,5,"NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500"
4,13,"NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000"
5,36,"NDCG: 0.491, Recall@10: 1.000, MAP@10: 0.278"
...,...,...
332,1379,"NDCG: 0.956, Recall@10: 1.000, MAP@10: 0.887"
336,1382,"NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000"
337,1385,"NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500"
338,1389,"NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000"


In [99]:
df = df.rename(columns={"Query ID": "query-id"})

In [25]:
# # Add after model initialization and before DataFrame creation
# def get_relevant_content_rank(query_text, retrieved_docs):
#     """Calculate cosine similarity rank for relevant content"""
#     query_vector = model.encode(query_text)
    
#     # Get text content for retrieved docs
#     doc_texts = [corpus.get(doc_id, "") for doc_id in retrieved_docs]
#     doc_vectors = model.encode(doc_texts)
    
#     # Calculate cosine similarities
#     similarities = []
#     for doc_vector in doc_vectors:
#         similarity = np.dot(query_vector, doc_vector) / (np.linalg.norm(query_vector) * np.linalg.norm(doc_vector))
#         similarities.append(similarity)
    
#     # Return average similarity as rank
#     return np.mean(similarities) if similarities else 0

# Modify the main loop where results are collected
# for query_id in selected_queries:
#     query_text = queries.get(query_id, "")
#     relevant_docs = set(qrels[qrels["query-id"] == query_id]["corpus-id"])
    
#     row = {"Query ID": query_id}
#     for i in range(iterations):
#         try:
#             retrieved_docs = safe_retrieve_top_k(query_text, k=10)
#             dcg, ndcg, recall, map_ = compute_metrics(retrieved_docs, relevant_docs)
#             row[f"Iteration {i+1}"] = f"NDCG: {ndcg:.3f}, Recall@10: {recall:.3f}, MAP@10: {map_:.3f}"
            
#             # Calculate relevant content rank
#             rank = get_relevant_content_rank(query_text, retrieved_docs)
#             row["relevant_content_rank"] = f"{rank:.3f}"
            
#         except Exception as e:
#             print(f"Error processing query {query_id}, iteration {i+1}: {e}")
#             row[f"Iteration {i+1}"] = "Error: Failed to retrieve results"
#             row["relevant_content_rank"] = "Error"
    
#     results.append(row)

# Create DataFrame with new column
# df = pd.DataFrame(results)

In [89]:
df

,Query ID,Metrics
0,query-id,"NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000"
1,1,"NDCG: 0.387, Recall@10: 1.000, MAP@10: 0.200"
2,3,"NDCG: 0.500, Recall@10: 1.000, MAP@10: 0.333"
3,5,"NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500"
4,13,"NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000"
...,...,...
335,1379,"NDCG: 0.956, Recall@10: 1.000, MAP@10: 0.887"
336,1382,"NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000"
337,1385,"NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500"
338,1389,"NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000"


In [27]:
# # After DataFrame creation
# # Reorder columns to put relevant_content_rank second
# columns = ['Query ID', 'relevant_content_rank']
# other_columns = [col for col in df.columns if col not in columns]
# df = df[columns + other_columns]

# # Verify column order
# print("New column order:", df.columns.tolist())
# print("\nFirst few rows with reordered columns:")
# print(df.head())

New column order: ['Query ID', 'relevant_content_rank', 'Iteration 1', 'Iteration 2', 'Iteration 3', 'deterministic']

First few rows with reordered columns:
  Query ID relevant_content_rank  \
1        1                 0.294   
2        3                 0.563   
3        5                 0.344   
4       13                 0.467   
5       36                 0.618   

                                    Iteration 1  \
1  NDCG: 0.387, Recall@10: 1.000, MAP@10: 0.200   
2  NDCG: 0.500, Recall@10: 1.000, MAP@10: 0.333   
3  NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500   
4  NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000   
5  NDCG: 0.491, Recall@10: 1.000, MAP@10: 0.278   

                                    Iteration 2  \
1  NDCG: 0.387, Recall@10: 1.000, MAP@10: 0.200   
2  NDCG: 0.500, Recall@10: 1.000, MAP@10: 0.333   
3  NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500   
4  NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000   
5  NDCG: 0.491, Recall@10: 1.000, MAP@10: 0.278   

           

In [28]:
df

,Query ID,relevant_content_rank,Iteration 1,Iteration 2,Iteration 3,deterministic
1,1,0.294,"NDCG: 0.387, Recall@10: 1.000, MAP@10: 0.200","NDCG: 0.387, Recall@10: 1.000, MAP@10: 0.200","NDCG: 0.387, Recall@10: 1.000, MAP@10: 0.200",
2,3,0.563,"NDCG: 0.500, Recall@10: 1.000, MAP@10: 0.333","NDCG: 0.500, Recall@10: 1.000, MAP@10: 0.333","NDCG: 0.500, Recall@10: 1.000, MAP@10: 0.333",
3,5,0.344,"NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500","NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500","NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500",
4,13,0.467,"NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000","NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000","NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000",
5,36,0.618,"NDCG: 0.491, Recall@10: 1.000, MAP@10: 0.278","NDCG: 0.491, Recall@10: 1.000, MAP@10: 0.278","NDCG: 0.491, Recall@10: 1.000, MAP@10: 0.278",
6,42,0.524,"NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500","NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500","NDCG: 0.631, Recall@10: 1.000, MAP@10: 0.500",
7,48,0.413,"NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000","NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000","NDCG: 0.000, Recall@10: 0.000, MAP@10: 0.000",
8,49,0.477,"NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000","NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000","NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000",
9,50,0.407,"NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000","NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000","NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000",
10,51,0.505,"NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000","NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000","NDCG: 1.000, Recall@10: 1.000, MAP@10: 1.000",


In [113]:
df.to_csv("evaluation_metrics_scifact_final.csv", index=False)

In [105]:
df1 = pd.read_csv("merged_results.csv")

In [106]:
columns = ["query-id"] + [col for col in df.columns if col != "query-id"]
df = df[columns]

In [107]:
merged_df = pd.merge(df1, df, on="query-id", how="inner") 

ValueError: You are trying to merge on int64 and object columns for key 'query-id'. If you wish to proceed you should use pd.concat